# 01 — Descarga de datos GBIF
Descarga occurrencias de Aves y Mamíferos en Bolivia (2000–2024) desde GBIF.

In [ ]:
import os
import time
from pygbif import occurrences as occ

# Credenciales GBIF — reemplazá con tus datos
os.environ['GBIF_USER']  = 'max142'
os.environ['GBIF_PWD']   = 'Balack34$'
os.environ['GBIF_EMAIL'] = 'barryallen4207@gmail.com'

os.makedirs('../data/raw/gbif', exist_ok=True)

print('✅ Listo')

In [ ]:
FILTROS = [
    'country = BO',
    'hasCoordinate = TRUE',
    'hasGeospatialIssue = FALSE',
    'year >= 2000',
    'year <= 2024',
]

key_aves = occ.download(['classKey = 212'] + FILTROS)[0]
key_mam  = occ.download(['classKey = 359'] + FILTROS)[0]

print(f'key_aves = "{key_aves}"')
print(f'key_mam  = "{key_mam}"')
print('\n⚠️  Guardá estos keys por si el notebook se interrumpe')

In [ ]:
# Si retomás desde aquí pegá los keys y descomentá:
# key_aves = 'PEGA-TU-KEY'
# key_mam  = 'PEGA-TU-KEY'

def esperar_y_descargar(key, nombre, carpeta='../data/raw/gbif'):
    print(f'⏳ [{nombre}] Esperando...')
    for i in range(1, 999):
        status = occ.download_meta(key)['status']
        print(f'   [{i:02d}] {status}')
        if status == 'SUCCEEDED':
            break
        if status in ('FAILED', 'KILLED'):
            print(f'❌ Error: {status}')
            return None
        time.sleep(30)

    occ.download_get(key, path=carpeta)
    zip_path = os.path.join(carpeta, f'{key}.zip')
    print(f'💾 Guardado: {zip_path}')
    return zip_path

zip_aves = esperar_y_descargar(key_aves, 'Aves')
zip_mam  = esperar_y_descargar(key_mam,  'Mamíferos')

print('\n✅ Descargas completas')

In [11]:
df_aves = pd.read_csv('../data/raw/gbif/0051889-260226173443078.csv', sep='\t', low_memory=False)

print(f'Total registros: {len(df_aves):,}')
print(f'\ntaxonRank:')
print(df_aves['taxonRank'].value_counts(dropna=False))
print(f'\nEjemplos de species:')
print(df_aves['species'].dropna().head(10).tolist())
print(f'\nNulos en species: {df_aves["species"].isna().sum():,} ({df_aves["species"].isna().mean()*100:.1f}%)')

Total registros: 1,385,784

taxonRank:
taxonRank
SPECIES       1368741
GENUS           13497
SUBSPECIES       2385
VARIETY           557
UNRANKED          407
FAMILY             91
FORM               89
CLASS              17
Name: count, dtype: int64

Ejemplos de species:
['Nothoprocta ornata', 'Elaenia albiceps', 'Coryphospingus cucullatus', 'Nothura darwinii', 'Myiodynastes maculatus', 'Synallaxis azarae', 'Myiodynastes maculatus', 'Phyllomyias uropygialis', 'Myiozetetes cayanensis', 'Nyctibius griseus']

Nulos en species: 13,688 (1.0%)


In [12]:
especies_unicas = df_aves[df_aves['taxonRank'] == 'SPECIES']['species'].dropna().unique()
print(f'Especies únicas a consultar en IUCN: {len(especies_unicas):,}')

Especies únicas a consultar en IUCN: 1,478


In [ ]:
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# La API de GBIF no devuelve categorías IUCN fiables.
# Usamos Wikidata SPARQL que sí las tiene para la mayoría de aves.

especies_unicas = df_aves[df_aves['taxonRank'] == 'SPECIES']['species'].dropna().unique()
print(f'Especies a consultar en Wikidata: {len(especies_unicas):,}')

LABEL_A_CODIGO = {
    'Least Concern':        'LC',
    'Near Threatened':      'NT',
    'Vulnerable':           'VU',
    'Endangered':           'EN',
    'Critically Endangered':'CR',
    'Extinct in the Wild':  'EW',
    'Extinct':              'EX',
    'Data Deficient':       'DD',
}

SPARQL_URL = 'https://query.wikidata.org/sparql'
HEADERS    = {'User-Agent': 'iucn-aves-bolivia/1.0'}

def consultar_batch(batch):
    """Consulta un batch de nombres de especies en Wikidata.
    Usa p:P141/ps:P141 + pq:P585 para recuperar todos los statements
    (incluyendo historicos) y toma el mas reciente por especie.
    Esto evita el problema de wdt:P141 que devuelve solo el statement
    'preferred' o todos los 'normal', pudiendo omitir evaluaciones actuales.
    """
    valores = ' '.join(f'"{e}"' for e in batch)
    query = f"""
SELECT ?taxonName ?iucnLabel ?date WHERE {{
  VALUES ?taxonName {{ {valores} }}
  ?taxon wdt:P225 ?taxonName .
  ?taxon p:P141 ?stmt .
  ?stmt ps:P141 ?iucn .
  OPTIONAL {{ ?stmt pq:P585 ?date }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" }}
}}
ORDER BY ?taxonName DESC(?date)
"""
    for intento in range(3):
        try:
            r = requests.get(SPARQL_URL, params={'query': query, 'format': 'json'},
                             headers=HEADERS, timeout=45)
            r.raise_for_status()
            filas = r.json()['results']['bindings']
            # Tomar el mas reciente por especie (primero tras ORDER BY DESC date)
            seen = {}
            for row in filas:
                name = row['taxonName']['value']
                if name not in seen:
                    seen[name] = LABEL_A_CODIGO.get(row['iucnLabel']['value'])
            return seen
        except Exception as e:
            if intento == 2:
                print(f'  Batch fallo: {e}')
                return {}
            time.sleep(5)

# Procesar en batches de 50 especies
BATCH_SIZE = 50
resultados = {}
batches = [list(especies_unicas[i:i+BATCH_SIZE]) for i in range(0, len(especies_unicas), BATCH_SIZE)]

for i, batch in enumerate(batches, 1):
    res = consultar_batch(batch)
    resultados.update(res)
    if i % 5 == 0 or i == len(batches):
        con_cat = sum(v is not None for v in resultados.values())
        print(f'  [{i}/{len(batches)} batches] {len(resultados)} consultadas, {con_cat} con categoria')
    time.sleep(1)  # respetar rate limit de Wikidata

# Armar DataFrame con todas las especies (incluyendo las sin categoria -> NaN)
df_iucn = pd.DataFrame([
    {'species': e, 'iucn_categoria': resultados.get(e)}
    for e in especies_unicas
])

print(f'''
=== RESULTADO ===
{df_iucn['iucn_categoria'].value_counts(dropna=False).to_string()}
''')

import os
os.makedirs('../data/raw/iucn', exist_ok=True)
df_iucn.to_csv('../data/raw/iucn/iucn_aves.csv', index=False)
print('Guardado en data/raw/iucn/iucn_aves.csv')